# 02 · Motion models and temporal evaluation

**NFL Player Trajectory Lab** · Reproducible real-data benchmark

Compare five physical references with a role-conditioned ridge model. The ridge model
learns six shared x/y vector weights for each role, using only training-game sufficient
statistics. Fixed regularization is declared in the protocol before evaluation.

This is a local experiment, not a Kaggle leaderboard score or a medal claim.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
LOCAL = ROOT / "artifacts" / "benchmark"
PUBLISHED = ROOT / "docs" / "results"
RESULTS = LOCAL if (LOCAL / "summary.json").exists() else PUBLISHED
READY = (RESULTS / "summary.json").exists()
if READY:
    summary = json.loads((RESULTS / "summary.json").read_text())
    eda = json.loads((RESULTS / "eda.json").read_text())
    protocol = json.loads((RESULTS / "protocol.json").read_text())
    display(Markdown("**Report source:** " + ("local benchmark artifacts" if RESULTS == LOCAL else "published reproducible experiment snapshot")))
else:
    display(Markdown(
        "Run `nfl benchmark` after the data audit to create the real-data results. "
        "No synthetic result is substituted here."
    ))

## The official metric

$$\mathrm{RMSE}=\sqrt{\frac{\sum_{i=1}^{N}[(\hat{x}_i-x_i)^2+(\hat{y}_i-y_i)^2]}{2N}}.$$

Coordinates are measured in yards. Every coordinate/frame receives equal weight.
ADE measures Euclidean displacement, FDE measures each trajectory's final displacement,
and p95 reports the error tail. They complement RMSE; they are different quantities.

In [ ]:
if READY:
    scores = pd.DataFrame(summary["models"])
    display(scores[["model", "coordinate_rmse_yards", "rmse_ci95_low", "rmse_ci95_high", "ade_frame_weighted_yards", "fde_trajectory_weighted_yards", "p95_displacement_yards"]].round(4))
    best = scores.iloc[0]
    display(Markdown(
        f"**Selected for further research:** `{best['model']}`. "
        f"RMSE is **{summary['improvement_vs_velocity_percent']:.1f}% lower** "
        "than constant velocity on these validation games."
    ))

In [ ]:
if READY:
    display(Image(filename=str(RESULTS / "benchmark.png")))

## Interpret the learned weights

The six basis vectors are last velocity × time, recent velocity × time,
acceleration × time²/2, and the vector to the landing point multiplied by normalized
time, normalized time², and normalized time³. Training-RMS scaling and ridge stabilize
the fit. Coordinates share coefficients, preserving rotation and translation equivariance.
An unseen role uses the global training fit. Correlated weights should be interpreted jointly.

In [ ]:
if READY:
    display(Image(filename=str(RESULTS / "coefficients.png")))
    latency = json.loads((RESULTS / "latency.json").read_text())
    display(pd.DataFrame([latency]))

## Watch trajectories and reproduce the experiment

Open `artifacts/benchmark/report.html` for the full offline report and animated field.
The example is the first validation play by game/play ID, selected independently of its errors.

A repeat of `nfl benchmark` verifies cached files and reuses completed weekly stages.
Changed inputs, model source, or dependencies invalidate corresponding checkpoints.
Interrupted stages are recomputed, and a corrupt artifact never counts as completed.

## What this result establishes

Game-cluster bootstrap intervals resample entire games 2,000 times, preserving player
and frame dependence within each sampled game. Paired intervals compare against
constant velocity. They describe this validation sample, not every future season.
Repeated model selection can overfit validation. Keep the holdout reserved until
model selection is locked.

## Where the baseline still struggles

Read the role and forecast-time slices rather than only the overall average.
These are development-validation diagnostics, not new holdout results. Longer
forecasts and defensive coverage motivate the interaction feature experiment.

In [ ]:
if READY:
    slices = pd.DataFrame(summary["slices"])
    display(slices.loc[slices["model"].eq("role_ridge"), ["dimension", "value", "rows", "coordinate_rmse_yards"]].round(4))

## Feature-family ablations

Run `nfl features` after the original benchmark. It preserves the original model,
prepares one week at a time, screens/scales on training games only, and evaluates
all challengers on exactly the same validation rows. Completed stages are hashed
and reused. The chart includes game-cluster confidence intervals; negative paired
RMSE differences versus role ridge favour a challenger.

The residual learner is intentionally interpretable. It is **not** a reproduced
winning temporal neural model. The feature bank and tests do not establish predictive
improvement until the real-data ablations below have completed.

In [ ]:
feature_local = ROOT / "artifacts/features/summary.json"
feature_published = PUBLISHED / "feature_summary.json"
feature_path = feature_local if feature_local.is_file() else feature_published
if feature_path.is_file():
    feature_summary = json.loads(feature_path.read_text())
    assert feature_summary["status"] == "passed"
    assert feature_summary["screening_split"] == "train"
    assert feature_summary["holdout_evaluation"] == "not_run"
    label = "Local experiment" if feature_path == feature_local else "Published experiment snapshot"
    display(Markdown(
        f"**Evidence:** {label}. "
        f"**Candidates:** {feature_summary['candidate_features']:,}."
    ))
    feature_scores = pd.DataFrame(feature_summary["models"])
    display(feature_scores[["model", "selected_features", "coordinate_rmse_yards", "ade_frame_weighted_yards", "fde_trajectory_weighted_yards", "delta_vs_role_ridge_ci95"]].round(4))
    feature_image = ROOT / "artifacts/features/benchmark.png" if feature_path == feature_local else PUBLISHED / "feature_benchmark.png"
    display(Image(filename=str(feature_image)))
else:
    display(Markdown(
        "**Real-data feature ablation: not run in this published snapshot.** "
        "The measured baseline above is unchanged; no challenger score is being claimed."
    ))